# Web Service Testing Notebook

This notebook helps you test the Python web service.

## Before You Start

1. Make sure the web service is running!
   - Either double-click `run.bat`
   - Or run: `python -m uvicorn main:app --reload`

2. The service should be available at `http://localhost:8000`

## How to Use This Notebook

- Click on a cell to select it
- Press `Shift+Enter` to run the cell
- Results appear below the cell
- Run cells in order from top to bottom

---
## Cell 1: Import Libraries

First, we import the libraries we need:
- `requests` - for making HTTP requests to our service
- `json` - for pretty-printing JSON responses

In [4]:
# Import the requests library
# This library lets us send HTTP requests (GET, POST, etc.) from Python
import requests

# Import json for pretty printing
import json

# Define the base URL of our service
# Change this if your service is running on a different port or machine
BASE_URL = "http://localhost:8000"

# Helper function to print JSON nicely
def print_json(data):
    """Pretty print a dictionary as formatted JSON."""
    print(json.dumps(data, indent=2))

print("Libraries imported successfully!")
print(f"Base URL: {BASE_URL}")

Libraries imported successfully!
Base URL: http://localhost:8000


---
## Cell 2: Test the Root Endpoint

Let's test the root endpoint (`/`) to make sure the service is running.

This is a **GET** request - we're just asking for information.

In [5]:
# Make a GET request to the root endpoint
# requests.get() sends a GET request and returns the response
response = requests.get(f"{BASE_URL}/")

# Print the status code
# 200 = OK (success)
# 404 = Not Found
# 500 = Server Error
print(f"Status Code: {response.status_code}")
print()

# Print the response body as JSON
# .json() converts the response body from JSON text to a Python dictionary
print("Response:")
print_json(response.json())

Status Code: 200

Response:
{
  "message": "Welcome to the Web Service",
  "documentation": "Visit /docs for interactive API documentation",
  "health_check": "Visit /health to check service status"
}


---
## Cell 3: Test the Health Check Endpoint

The health check endpoint (`/health`) tells us if the service is healthy.

This is useful for:
- Monitoring systems
- Load balancers
- Quick "is it working?" checks

In [6]:
# Test the health check endpoint
response = requests.get(f"{BASE_URL}/health")

print(f"Status Code: {response.status_code}")
print()
print("Response:")
print_json(response.json())

# Check if the service is healthy
if response.json().get("status") == "healthy":
    print("\n[OK] Service is healthy!")
else:
    print("\n[WARNING] Service may have issues!")

Status Code: 200

Response:
{
  "status": "healthy",
  "message": "Service is running normally"
}

[OK] Service is healthy!


---
## Cell 4: Test the Process Endpoint (POST Request)

Now let's test the main endpoint (`/api/process`).

This is a **POST** request - we're sending data to be processed.

We need to send JSON data in the request body.

In [7]:
# Define the data we want to send
# This must match the GenericRequest model in models.py
request_data = {
    "action": "test",              # Required: what action to perform
    "data": {                       # Optional: additional data
        "message": "Hello from Jupyter!",
        "number": 42
    },
    "request_id": "notebook-test-001"  # Optional: ID to track this request
}

print("Sending this request:")
print_json(request_data)
print()

# Make a POST request
# requests.post() sends a POST request
# json=request_data automatically:
#   1. Converts the dictionary to JSON
#   2. Sets the Content-Type header to application/json
response = requests.post(
    f"{BASE_URL}/api/process",
    json=request_data
)

print(f"Status Code: {response.status_code}")
print()
print("Response:")
print_json(response.json())

Sending this request:
{
  "action": "test",
  "data": {
    "message": "Hello from Jupyter!",
    "number": 42
  },
  "request_id": "notebook-test-001"
}

Status Code: 200

Response:
{
  "success": true,
  "message": "Successfully processed action: test",
  "result": {
    "received_action": "test",
    "received_data": {
      "message": "Hello from Jupyter!",
      "number": 42
    },
    "processed": true
  },
  "request_id": "notebook-test-001",
  "timestamp": "2026-01-12T17:23:11.650229"
}


---
## Cell 5: Test with Minimal Data

Let's send a request with only the required fields.

The `action` field is required. The `data` and `request_id` fields are optional.

In [8]:
# Minimal request - only the required 'action' field
minimal_request = {
    "action": "minimal_test"
}

print("Sending minimal request:")
print_json(minimal_request)
print()

response = requests.post(
    f"{BASE_URL}/api/process",
    json=minimal_request
)

print(f"Status Code: {response.status_code}")
print()
print("Response:")
print_json(response.json())

Sending minimal request:
{
  "action": "minimal_test"
}

Status Code: 200

Response:
{
  "success": true,
  "message": "Successfully processed action: minimal_test",
  "result": {
    "received_action": "minimal_test",
    "received_data": {},
    "processed": true
  },
  "request_id": null,
  "timestamp": "2026-01-12T17:23:20.791954"
}


---
## Cell 6: Test Error Handling (Missing Required Field)

What happens if we send invalid data? Let's try sending a request without the required `action` field.

FastAPI should return a `422 Unprocessable Entity` error with details about what's wrong.

In [9]:
# Invalid request - missing the required 'action' field
invalid_request = {
    "data": {"key": "value"}
    # Notice: no 'action' field!
}

print("Sending invalid request (missing 'action'):")
print_json(invalid_request)
print()

response = requests.post(
    f"{BASE_URL}/api/process",
    json=invalid_request
)

print(f"Status Code: {response.status_code}")
print()
print("Response (error details):")
print_json(response.json())

Sending invalid request (missing 'action'):
{
  "data": {
    "key": "value"
  }
}

Status Code: 422

Response (error details):
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "action"
      ],
      "msg": "Field required",
      "input": {
        "data": {
          "key": "value"
        }
      }
    }
  ]
}


---
## Cell 7: Test the Echo Endpoint

The service also has an echo endpoint that demonstrates URL path parameters.

Try different messages!

In [10]:
# Test the echo endpoint with a message
message = "hello"

response = requests.get(f"{BASE_URL}/api/echo/{message}")

print(f"Sent message: '{message}'")
print(f"Status Code: {response.status_code}")
print()
print("Response:")
print_json(response.json())

Sent message: 'hello'
Status Code: 200

Response:
{
  "you_said": "hello",
  "echo": "HELLO"
}


---
## Cell 8: Making Multiple Requests

In real scenarios, you might want to make multiple requests.

Here's how to loop through different test cases.

In [11]:
# Define multiple test cases
test_cases = [
    {"action": "create", "data": {"name": "Item 1"}},
    {"action": "update", "data": {"id": 1, "name": "Updated Item"}},
    {"action": "delete", "data": {"id": 1}},
]

# Loop through and test each one
for i, test_data in enumerate(test_cases, 1):
    print(f"=== Test Case {i} ===")
    print(f"Action: {test_data['action']}")
    
    response = requests.post(
        f"{BASE_URL}/api/process",
        json=test_data
    )
    
    result = response.json()
    print(f"Success: {result['success']}")
    print(f"Message: {result['message']}")
    print()

=== Test Case 1 ===
Action: create
Success: True
Message: Successfully processed action: create

=== Test Case 2 ===
Action: update
Success: True
Message: Successfully processed action: update

=== Test Case 3 ===
Action: delete
Success: True
Message: Successfully processed action: delete



---
## Cell 9: Reusable Function for Testing

Here's a helper function you can use to quickly test different requests.

In [12]:
def test_process(action, data=None, request_id=None):
    """
    Send a test request to the process endpoint.
    
    Args:
        action: The action to perform (required)
        data: Optional dictionary of additional data
        request_id: Optional ID to track the request
    
    Returns:
        The response JSON as a dictionary
    """
    request_data = {"action": action}
    
    if data:
        request_data["data"] = data
    
    if request_id:
        request_data["request_id"] = request_id
    
    response = requests.post(
        f"{BASE_URL}/api/process",
        json=request_data
    )
    
    return response.json()

# Example usage:
result = test_process(
    action="calculate",
    data={"x": 10, "y": 20},
    request_id="calc-001"
)

print("Result:")
print_json(result)

Result:
{
  "success": true,
  "message": "Successfully processed action: calculate",
  "result": {
    "received_action": "calculate",
    "received_data": {
      "x": 10,
      "y": 20
    },
    "processed": true
  },
  "request_id": "calc-001",
  "timestamp": "2026-01-12T17:24:05.007260"
}


---
## Summary

You've learned how to:

1. **Import libraries** - `requests` for HTTP calls, `json` for formatting
2. **Make GET requests** - For reading data (`requests.get()`)
3. **Make POST requests** - For sending data (`requests.post()`)
4. **Send JSON data** - Using the `json=` parameter
5. **Read responses** - Using `.status_code` and `.json()`
6. **Handle errors** - Check status codes and error messages

## Next Steps

- Try modifying the `main.py` file to add new endpoints
- Create new test cases in this notebook
- Visit `http://localhost:8000/docs` for interactive testing